### **1. Load Dependencies**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import mlflow

E:\ML\RoadMap\Log-Classification-Hybrid-Framework\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **2. Load Data**

In [2]:
df = pd.read_csv("training/datasets/synthetic_logs.csv")
df

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert
...,...,...,...,...,...
2405,2025-08-13 07:29:25,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert
2406,1/11/2025 5:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert
2407,2025-08-03 03:07:47,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert


In [3]:
df = df.drop('timestamp',axis=1)

In [4]:
df.head()

,source,log_message,target_label,complexity
0,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


### **3. Preprocessing**

#### **3.1 Handle Missing values**

In [5]:
df.isnull().sum()

source          0
log_message     0
target_label    0
complexity      0
dtype: int64

#### **3.2 Handle wrong format values**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   source        2410 non-null   object
 1   log_message   2410 non-null   object
 2   target_label  2410 non-null   object
 3   complexity    2410 non-null   object
dtypes: object(4)
memory usage: 75.4+ KB


In [7]:
#### WE WILL CREATE EMBEDDINGS FOR THE LOG_MESSAGE COLUMN

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(df['log_message'].tolist())
print(embeddings.shape)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4652.61it/s]


(2410, 384)


In [8]:
embeddings[:1]

array([[-1.02939621e-01,  3.35459262e-02, -2.20260601e-02,
         1.55105593e-03, -9.86920949e-03, -1.78956255e-01,
        -6.34409785e-02, -6.01761676e-02,  2.81108841e-02,
         5.99620454e-02, -1.72618646e-02,  1.43367471e-03,
        -1.49560004e-01,  3.15285777e-03, -5.66031076e-02,
         2.71685384e-02, -1.49890380e-02, -3.54037546e-02,
        -3.62936258e-02, -1.45410234e-02, -5.61493542e-03,
         8.75539109e-02,  4.55120429e-02,  2.50963680e-02,
         1.00187706e-02,  1.24267014e-02, -1.39923617e-01,
         7.68696442e-02,  3.14095207e-02, -4.15248703e-03,
         4.36902680e-02,  1.71250310e-02, -8.00951496e-02,
         5.74005917e-02,  1.89091805e-02,  8.55262280e-02,
         3.96399237e-02, -1.34371832e-01, -1.44364929e-03,
         3.06707853e-03,  1.76854029e-01,  4.44886740e-03,
        -1.69275180e-02,  2.24266723e-02, -4.35049348e-02,
         6.09028572e-03, -9.98169184e-03, -6.23973049e-02,
         1.07372776e-02, -6.04895549e-03, -7.14660808e-0

#### **3.3 Handle wrong values**

#### **3.4 Handle duplicates**

### **4. Model Building**

#### **4.1 We will use unsupervised models to identify the patterns**

In [9]:
from sklearn.cluster import DBSCAN # DBSCAN is a algo like KMeans clustering

uns_model = DBSCAN(eps=0.2,min_samples=5,metric='cosine')

clusters = uns_model.fit_predict(embeddings)

df['cluster'] = clusters
df.head()

,source,log_message,target_label,complexity,cluster
0,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,-1
3,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [10]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [11]:
df.cluster.unique()

array([ 0,  1, -1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 19, 13, 14,
       15, 16, 17, 18, 22, 20, 21, 26, 25, 23, 24, 30, 31, 27, 28, 29],
      dtype=int64)

In [12]:
cluster_counts = df.cluster.value_counts()
large_clusters = cluster_counts[cluster_counts>10].index


for cluster in large_clusters:
    print(f"Cluster {cluster} :")
    print(df[df['cluster']==cluster]['log_message'].head(5))
    print()


Cluster 0 :
0    nova.osapi_compute.wsgi.server [req-b9718cd8-f...
3    nova.osapi_compute.wsgi.server [req-4895c258-b...
4    nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...
5    nova.osapi_compute.wsgi.server [req-f0bffbc3-5...
9    nova.osapi_compute.wsgi.server [req-2bf7cfee-a...
Name: log_message, dtype: object

Cluster -1 :
2             Unauthorized access to data was attempted
34    Suspicious login activity detected from 192.16...
43    Abnormal system behavior on server 40, potenti...
51    Server 4 restarted without warning during data...
55    Server 34 crashed unexpectedly while syncing data
Name: log_message, dtype: object

Cluster 4 :
8     nova.compute.claims [req-a07ac654-8e81-416d-bf...
26    nova.compute.claims [req-d6986b54-3735-4a42-90...
40    nova.compute.claims [req-72b4858f-049e-49e1-b3...
58    nova.compute.claims [req-5c8f52bd-8e3c-41f0-95...
61    nova.compute.claims [req-d38f479d-9bb9-4276-96...
Name: log_message, dtype: object

Cluster 10 :
27     User U

In [13]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (out|in)." : "User Action",
        r"Account with ID .* created by .*.": "User Action",
        r"Backup (started|ended) at .*":"System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*":"System Notification",
        r"Disk cleanup completed successfully.":"System Notification",
        r"System reboot initiated by user User.*":"System Notification",
    }

    for regex_pattern,label in regex_patterns.items():
        if re.search(regex_pattern,log_message):
            return label
    return None

msgs = [
"User User685 logged out.",
"User User395 logged in.",
"Backup started at 2025-02-15 20:00:19."
"Backup ended at 2025-08-08 13:06:23",
"Backup completed successfully.",
"File data_1206.csv uploaded successfully by user ..."
"File data_1503.csv uploaded successfully by user sdf",
"Disk cleanup completed successfully.",
"System reboot initiated by user User471.",
"Account with ID 2520 created by User546.",
'nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118	HTTP Status	bert	0'
]
for msg in msgs:
    print(msg,classify_with_regex(msg))

User User685 logged out. User Action
User User395 logged in. User Action
Backup started at 2025-02-15 20:00:19.Backup ended at 2025-08-08 13:06:23 System Notification
Backup completed successfully. System Notification
File data_1206.csv uploaded successfully by user ...File data_1503.csv uploaded successfully by user sdf System Notification
Disk cleanup completed successfully. System Notification
System reboot initiated by user User471. System Notification
Account with ID 2520 created by User546. User Action
nova.osapi_compute.wsgi.server [req-b9718cd8-f65e-49cc-8349-6cf7122af137 113d3a99c3da401fbd62cc2caa5b96d2 54fadb412c4e40cdbaed9335e4c35a9e - - -] 10.11.10.1 "GET /v2/54fadb412c4e40cdbaed9335e4c35a9e/servers/detail HTTP/1.1" status: 200 len: 1893 time: 0.2675118	HTTP Status	bert	0 None


In [14]:
largest_clusters = cluster_counts[:4].index  # this include the cluster with the most num of elements after clustering

for cluster in largest_clusters:
    print(f"Cluster {cluster} :")
    print(df[df['cluster']==cluster]['log_message'].head(5))
    print()

Cluster 0 :
0    nova.osapi_compute.wsgi.server [req-b9718cd8-f...
3    nova.osapi_compute.wsgi.server [req-4895c258-b...
4    nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...
5    nova.osapi_compute.wsgi.server [req-f0bffbc3-5...
9    nova.osapi_compute.wsgi.server [req-2bf7cfee-a...
Name: log_message, dtype: object

Cluster -1 :
2             Unauthorized access to data was attempted
34    Suspicious login activity detected from 192.16...
43    Abnormal system behavior on server 40, potenti...
51    Server 4 restarted without warning during data...
55    Server 34 crashed unexpectedly while syncing data
Name: log_message, dtype: object

Cluster 4 :
8     nova.compute.claims [req-a07ac654-8e81-416d-bf...
26    nova.compute.claims [req-d6986b54-3735-4a42-90...
40    nova.compute.claims [req-72b4858f-049e-49e1-b3...
58    nova.compute.claims [req-5c8f52bd-8e3c-41f0-95...
61    nova.compute.claims [req-d38f479d-9bb9-4276-96...
Name: log_message, dtype: object

Cluster 10 :
27     User U

In [15]:
df['regex_label'] = df['log_message'].apply(lambda x: classify_with_regex(x))
df

,source,log_message,target_label,complexity,cluster,regex_label
0,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,-1,None
3,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None
...,...,...,...,...,...,...
2405,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,None
2406,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,6,None
2407,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,None
2408,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [16]:
df['regex_label'].notnull().sum()

500

In [17]:
# We have successfully classified 500 out of 2k notifications
df_no_regex = df[df['regex_label'].isnull()]
df_no_regex

,source,log_message,target_label,complexity,cluster,regex_label
0,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,-1,None
3,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None
...,...,...,...,...,...,...
2405,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,None
2406,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,6,None
2407,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,None
2408,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [18]:
# lets check all the clusters containing User Action and system notification are missing in the df_no_regex (3,7,8,10,12,13,21) 
df_no_regex.cluster.unique()

array([ 0,  1, -1,  2,  4,  5,  6,  9, 11, 19, 14, 15, 16, 18, 22, 20, 26,
       25, 23, 24, 30, 31, 27, 28, 29], dtype=int64)

In [19]:
# great classfiy with regex have done its job
# no we will have to do classification with a model
# client said that log_messages with source LegacyCRM have few messages. so we will classify them using an LLM.Therefore we will make a new dataset without LegacyCRM to train the model
df_no_legacy =  df_no_regex[df_no_regex.source != "LegacyCRM"]
df_no_legacy



,source,log_message,target_label,complexity,cluster,regex_label
0,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,-1,None
3,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None
...,...,...,...,...,...,...
2405,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,None
2406,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,6,None
2407,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,None
2408,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [20]:
df_no_legacy.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [21]:
# successfully removed LegacyCRM rows.
df_no_legacy.cluster.unique()

array([ 0,  1, -1,  2,  4,  5,  6,  9, 11, 19, 14, 15, 16, 18, 22, 20, 26,
       25, 23, 24, 30, 31, 27, 28, 29], dtype=int64)

In [22]:
df_no_legacy.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'Resource Usage'], dtype=object)

In [23]:
# in a real life scenario we will have to select which cluster belong to which category.
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X = encoder.encode(df_no_legacy['log_message'].to_list())
y = df_no_legacy['target_label']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6866.51it/s]


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB,GaussianNB


model_params = {
    
        'svm':{
            'model':SVC(),
            'params':{
                'C':[1,2,3,4,5,7,10,11],
                'kernel':['linear', 'poly', 'rbf', 'sigmoid']
            }
        },

        'random_forest':{
            'model':RandomForestClassifier(),
            'params':{
                'n_estimators':[30,40,50,100,120,150],
                'criterion': ['gini', 'entropy', 'log_loss'],
            }
        },
        'logistic_regression':{
            'model':LogisticRegression(),
            'params':{
                'C':[1,2,3,4,5,6,7,10,11],
                'solver':['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
            }
        },     
        'gaussian_nb':{
            'model':GaussianNB(),
            'params':{
                'priors':[None],
                'var_smoothing': [1e-9]
            }
        },
}



In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

analytics = []
for model_name,mp in model_params.items():
    clf = GridSearchCV(estimator=mp['model'],param_grid=mp['params'],cv=5,return_train_score=False)
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    tmp = [model_name,clf.best_params_,clf.best_score_,clf,report]
    analytics.append(tmp)

In [30]:
analytics

[['svm',
  {'C': 2, 'kernel': 'poly'},
  0.9984990566303399,
  GridSearchCV(cv=5, estimator=SVC(),
               param_grid={'C': [1, 2, 3, 4, 5, 7, 10, 11],
                           'kernel': ['linear', 'poly', 'rbf', 'sigmoid']}),
  {'Critical Error': {'precision': 0.9795918367346939,
    'recall': 1.0,
    'f1-score': 0.9896907216494846,
    'support': 48.0},
   'Error': {'precision': 1.0,
    'recall': 0.9787234042553191,
    'f1-score': 0.989247311827957,
    'support': 47.0},
   'HTTP Status': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 304.0},
   'Resource Usage': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 49.0},
   'Security Alert': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 123.0},
   'accuracy': 0.9982486865148862,
   'macro avg': {'precision': 0.9959183673469388,
    'recall': 0.9957446808510639,
    'f1-score': 0.9957876066954883,
    'support': 571.0},
   'weighted avg': {'precis

In [32]:
no_legacy_target_cls = df_no_legacy.target_label.unique()

In [37]:
import mlflow

mlflow.set_experiment('Log_Classification')
mlflow.set_tracking_uri('http://localhost:5000')

for ex in analytics:
    with mlflow.start_run(run_name=ex[0]):
        mlflow.log_param('model',ex[0])
        mlflow.log_params(ex[1])
        mlflow.log_metric('accuracy',ex[2])
        for clz in no_legacy_target_cls:
            mlflow.log_metric(f'precision_{clz}',ex[4][clz]['precision'])
            mlflow.log_metric(f'recall_{clz}',ex[4][clz]['recall'])
            mlflow.log_metric(f'f1_score_{clz}',ex[4][clz]['f1-score'])



2026/04/29 19:01:47 INFO mlflow.tracking.fluent: Experiment with name 'Log_Classification' does not exist. Creating a new experiment.


🏃 View run svm at: http://localhost:5000/#/experiments/2/runs/f49144fe7bf54da0b219a84cdb7f9e21
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run random_forest at: http://localhost:5000/#/experiments/2/runs/04052899446d41cf8ae9d5b2780c9a00
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run logistic_regression at: http://localhost:5000/#/experiments/2/runs/1e1a1442ac9144829aeff97fab96a3c9
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run gaussian_nb at: http://localhost:5000/#/experiments/2/runs/c7a8c72db6e64dacb93356c48aefd28c
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [38]:
# from the anlytics SVM seems to be the best performing model
analytics[0]

['svm',
 {'C': 2, 'kernel': 'poly'},
 0.9984990566303399,
 GridSearchCV(cv=5, estimator=SVC(),
              param_grid={'C': [1, 2, 3, 4, 5, 7, 10, 11],
                          'kernel': ['linear', 'poly', 'rbf', 'sigmoid']}),
 {'Critical Error': {'precision': 0.9795918367346939,
   'recall': 1.0,
   'f1-score': 0.9896907216494846,
   'support': 48.0},
  'Error': {'precision': 1.0,
   'recall': 0.9787234042553191,
   'f1-score': 0.989247311827957,
   'support': 47.0},
  'HTTP Status': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 304.0},
  'Resource Usage': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 49.0},
  'Security Alert': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 123.0},
  'accuracy': 0.9982486865148862,
  'macro avg': {'precision': 0.9959183673469388,
   'recall': 0.9957446808510639,
   'f1-score': 0.9957876066954883,
   'support': 571.0},
  'weighted avg': {'precision': 0.9982844276064191,
   're

In [40]:
import joblib

model = SVC(C=2,kernel='poly',probability=True)
model.fit(X_train,y_train)
joblib.dump(model,'./models/log_clf.joblib')

['./models/log_clf.joblib']